# STFT 心音去噪 — Demo

从上到下 Run all。三条臂，两个对比：

| 臂 | 训练干净心音 | 模型 |
|---|---|---|
| 1 device | device（subjects 1/2/4/5） | 简单 STFT 2D U-Net，约 274k |
| 2 扩展 | 每 epoch 50% device + 50% CirCor | 同一个 STFT 2D U-Net |
| 3 waveform | device（subjects 1/2/4/5） | CleanUNet，约 564k |

臂1 vs 臂2 = 扩展训练集有没有用。臂1 vs 臂3 = 两套去噪系统的效果对比。
三条臂都只输入一个 noisy chest waveform；STFT 网络的两个 Conv2d channel 是 `[Re, Im]`。
三条臂使用完全相同的固定 epoch/optimizer-step 预算，跑满后只比较 `final.pt`；没有 validation、early stopping 或 plateau scheduler。

**先确认 Runtime → Change runtime type → GPU。**

> **建议提前跑第 1–2 节**（约 20 分钟）。CirCor 池会存到你的 Drive，之后每次 session 几秒钟就复制回来，
> demo 当天这两节直接跳过。


## 1. 代码 + Drive（1 分钟）


In [ ]:
!nvidia-smi -L || echo '⚠️ 没有 GPU：Runtime → Change runtime type → GPU'

import os, sys, shutil, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

def run_checked(*args, cwd=None, tail=None):
    """Run one command, show an optional tail, and always propagate failure."""
    capture = tail is not None
    result = subprocess.run(
        [str(arg) for arg in args], cwd=cwd, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if capture and result.stdout:
        print('\n'.join(result.stdout.rstrip().splitlines()[-tail:]))
    result.check_returncode()

# Drive 上放缓存的地方（会自动创建）
CACHE = Path('/content/drive/MyDrive/stft_cache')
CACHE.mkdir(parents=True, exist_ok=True)

REPO = Path('/content/STFT_MIC_DENOISE')
if not REPO.exists():
    run_checked('git', 'clone', '-q', 'https://github.com/ArtieXu/STFT_MIC_DENOISE.git', REPO)
else:
    run_checked('git', '-C', REPO, 'pull', '--ff-only')
%cd {REPO}
sys.path.insert(0, str(REPO))
os.environ['PYTHONPATH'] = str(REPO)
run_checked(sys.executable, '-m', 'pip', 'install', '-q', 'soundfile')

import torch
print('torch', torch.__version__, '| GPU', torch.cuda.is_available(), '| 缓存目录', CACHE)


## 2. 数据（第一次约 15 分钟，之后 10 秒）

device 心音和 noise-only 窗口随 repo 一起下来了。训练样本始终是 `noisy = clean + scaled_noise`。
训练 clean 固定为 `heart_aw1`、`heart_bw1`、`heart_aw2`、`heart_bw2`、`heart_aw4`、`heart_bw4`、`heart_bw5`；训练 noise 固定为 `noise1/2/3/5`。
四个 noise 录音每 epoch 严格 1:1:1:1，并与 clean 来源独立混配；扩展臂的 2×4 种 clean-source×noise 组合等频。不要求 clean 与 noise 的编号相同。subject 6 的 clean 与 `noise6` 只做最终合成测试，`heart_w6` 只做真实走路定性展示。
要额外准备的是两样：

- **CirCor 池** —— 下 449 MB 原始包再采样，产出一个几百 MB 的 npz。**建好后存进 Drive，以后直接复用。**
- **真实走路录音** —— 15 MB，定性谱图要用。

CirCor 采样只在 TSV 标注段内取窗、丢掉 `Murmur=Unknown` 的受试者，并先用官方 `Additional ID` 把同一个人的两次采集合并后再切分 —— 那些段里官方列的噪声源是
听诊器摩擦、说话、小孩哭笑，当干净目标喂进去等于教模型输出噪声。训练时只使用 CirCor 池内的 `train` split。


In [ ]:
POOL = Path('data/circor/circor_pool_4khz_2s_v2.npz')
CACHED = CACHE / POOL.name
POOL.parent.mkdir(parents=True, exist_ok=True)

if CACHED.is_file() and not POOL.is_file():
    print('从 Drive 复用 CirCor 池…')
    shutil.copy(CACHED, POOL)

if not POOL.is_file():
    print('第一次：下载 + 采样 CirCor，约 15 分钟')
    run_checked('wget', '-c', '-q', '--show-progress', '-O', '/content/circor.zip', 'https://physionet.org/content/circor-heart-sound/get-zip/1.0.3/')
    run_checked('unzip', '-q', '-o', '/content/circor.zip', '-d', '/content')
    run_checked(sys.executable, 'scripts/build_circor_pool.py', '--root', '/content/circor-heart-sound-1.0.3', cwd=REPO)
    shutil.copy(POOL, CACHED)          # 存进 Drive，下次跳过整段
    print('已缓存到', CACHED)

run_checked(sys.executable, 'scripts/fetch_device_data.py', '--include-test-real', cwd=REPO, tail=2)
print('CirCor 池:', POOL.is_file(), '|', round(POOL.stat().st_size/2**20), 'MB')


## 3. 自检（2 分钟）—— 数据体检、单测、最小训练跑通整条路


In [ ]:
run_checked(sys.executable, 'scripts/audit_pools.py', cwd=REPO, tail=12)
run_checked(sys.executable, 'scripts/check_frequency_pipeline.py', cwd=REPO, tail=3)
run_checked(sys.executable, 'train_frequency.py', '--smoke', '--device', 'cpu', '--no_circor', cwd=REPO, tail=2)


## 4. 先测速，再决定训练规模

这一格量出这台机器上一步实际多少毫秒，推算三条臂总共要多久 —— 比训到一半发现来不及好。
它还会告诉你瓶颈在数据还是在计算。


In [ ]:
EPOCHS, SAMPLES, BATCH, SEED = 60, 20000, 16, 2026
assert SAMPLES % BATCH == 0, '固定预算要求 samples_per_epoch 能被 batch_size 整除'
assert SAMPLES % 8 == 0, '扩展臂要求 2 个 clean 来源 × 4 个 noise 录音的组合严格等频'
assert BATCH % 8 == 0, '每个 batch 也必须覆盖完整的 2×4 采样周期'
STEPS_PER_EPOCH = SAMPLES // BATCH
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
print('每条臂固定预算:', EPOCHS, 'epochs ×', STEPS_PER_EPOCH, 'steps =', TOTAL_STEPS, 'optimizer steps')

run_checked(sys.executable, 'scripts/benchmark.py', '--epochs', EPOCHS, '--samples_per_epoch', SAMPLES, '--batch_size', BATCH, cwd=REPO)


上面报的总时长如果超出你的预算，改上一格的三个数再跑一次。参考档位：

| 设置 | 相对计算量 |
|---|---|
| `60, 20000, 16` | 1.0（充分训练） |
| `30, 10000, 16` | 0.25 |
| `20, 8000, 32` | 0.13（够看出谱图差异） |

**三条臂必须用同一组数值** —— 否则比的是训练预算，不是你想测的东西。
训练不会根据 validation 提前停止或改变学习率；跑满固定预算后的模型写入 `final.pt`。
（`--num_workers` 默认尝试 2；若环境禁用共享内存，会明确降为 0，避免 worker 在后续 epoch 重复旧采样配方。）


## 5. 训练三条臂


In [ ]:
COMMON_ARGS = ['--epochs', EPOCHS, '--samples_per_epoch', SAMPLES, '--batch_size', BATCH, '--seed', SEED]
CHECKPOINT_CACHE = CACHE / 'checkpoints'
CHECKPOINT_CACHE.mkdir(parents=True, exist_ok=True)

def train_and_save(name, *extra_args):
    local = Path('checkpoints') / name
    cached = CHECKPOINT_CACHE / name
    # 清掉这一臂的旧结果；否则本次失败后旧 final.pt 会造成假成功。
    shutil.rmtree(local, ignore_errors=True)
    shutil.rmtree(cached, ignore_errors=True)
    run_checked(sys.executable, 'train_frequency.py', *COMMON_ARGS, *extra_args,
                '--output_dir', local, cwd=REPO)
    final = local / 'final.pt'
    if not final.is_file():
        raise RuntimeError(f'{name} finished without {final}')
    shutil.copytree(local, cached)
    print(name, 'validated checkpoint -> Drive')

train_and_save('device_only', '--no_circor')
train_and_save('combined')
train_and_save('waveform', '--no_circor', '--arch', 'cleanunet')


In [ ]:
# 每条臂完成后已经立即保存到 Drive；新 Colab session 会恢复并严格验证。
from src.arms import load_arm
for name, role in [('device_only', 'device_only'), ('combined', 'combined'), ('waveform', 'waveform')]:
    cached = CHECKPOINT_CACHE / name
    local = Path('checkpoints') / name
    if not (cached / 'final.pt').is_file():
        raise FileNotFoundError(f'缺少已完成的缓存：{cached / "final.pt"}')
    shutil.rmtree(local, ignore_errors=True)
    shutil.copytree(cached, local)
    load_arm(local / 'final.pt', torch.device('cpu'), expected_role=role)
    print(name, 'validated', local / 'final.pt')


## 6. 定量结果

评测集在脚本里现场构建：只使用完整留出的 device subject 6 clean 与 `noise6`，七个固定 SNR 档各 200 样本，固定 seed。第一次查看 subject 6 结果前必须冻结模型、loss、epoch 预算和评测 recipe；不能看完 subject 6 后再调参或另选 checkpoint。
三个模型吃到**逐字节相同**的混合，报告相对 raw noisy input 的去噪增益和系统间描述性差值。

重叠窗口不是独立受试者，因此这里不输出误导性的 window-level CI 或 p 值。


In [ ]:
run_checked(sys.executable, 'scripts/compare_datasets.py',
            '--device_only', 'checkpoints/device_only/final.pt',
            '--combined', 'checkpoints/combined/final.pt',
            '--waveform', 'checkpoints/waveform/final.pt', cwd=REPO)


In [ ]:
from IPython.display import Image, Audio, display
display(Image('outputs/comparison/comparison.png'))


## 7. 谱图 ← demo 上主要放这两张


In [ ]:
run_checked(sys.executable, 'scripts/make_demo_figures.py', '--all', '--wav',
            '--device_only', 'checkpoints/device_only/final.pt',
            '--combined', 'checkpoints/combined/final.pt',
            '--waveform', 'checkpoints/waveform/final.pt', cwd=REPO)


In [ ]:
display(Image('outputs/demo/demo_synthetic_grid.png'))


In [ ]:
display(Image('outputs/demo/demo_real_walking.png'))


In [ ]:
# 听一下真实走路录音（比看更有说服力）
for name in ['chest', 'device_only', 'combined', 'waveform']:
    p = Path(f'outputs/demo/wav_real/{name}.wav')
    if p.exists():
        print(name); display(Audio(str(p)))


## 8. 现场可能被问到的三件事

**「输入到底是什么？」** 一个 noisy chest waveform。STFT 后只拆成实部和虚部两个平面，
没有第二个麦克风、reference、解析谱减或固定 bandpass。

**「真实数据上呢？」** 第 7 节第二张图是真实走路录音，没有任何合成。但要说清楚：
没有干净参考所以**算不出数字**，只能听和看。图使用 test-only subject 6 预先固定的 0–10 秒片段，
不会自动挑表现最好的一段。

**「差异可信吗？」** subject 6 与训练完全隔离，但仍只有一位测试受试者；表中差值是描述性的。
正式结论至少用三组 matched seeds 重跑三条臂，不能把重叠 windows 当成独立人群样本。
